# 📉 Análise de Devoluções e Ralo Financeiro

## Objetivo de Negócio
Após descobrirmos que nossa margem de preços na ponta é altíssima (>50%), precisamos investigar se estamos perdendo dinheiro através de **Devoluções**. 

Esta análise responde:
1. **Motivos principais:** Por que os clientes estão devolvendo nossos produtos?
2. **Os Verdadeiros Vilões:** Quais SKUs têm alto volume de devolução (prejuízo de frete/estorno)?

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

con = duckdb.connect()
df_dev = con.execute("SELECT * FROM '../data/processed/fato_devolucoes.parquet'").df()
df_vendas = con.execute("SELECT * FROM '../data/processed/fato_vendas.parquet'").df()

print(f"Foram carregados {len(df_dev):,} itens devolvidos em todo o histórico válido.")

### 1. Quais são os motivos principais de devolução?

In [ ]:
motivos = df_dev.groupby('return_reason')['return_quantity'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=motivos, y='return_reason', x='return_quantity', palette='Reds_r')
plt.title('Volume de Itens Devolvidos por Motivo', fontsize=14)
plt.xlabel('Quantidade de Itens')
plt.ylabel('Motivo')
plt.show()

### 2. Identificando os SKUs ofensores (Os Vilões)
Vamos calcular a Taxa de Devolução por Produto: (Quantidade Devolvida / Quantidade Vendida).

In [ ]:
# Somando vendas totais por SKU
vendas_sku = df_vendas.groupby('sku')['quantity'].sum().reset_index().rename(columns={'quantity': 'qty_vendida'})
# Somando devoluções por SKU
dev_sku = df_dev.groupby(['sku', 'product_name']).agg({
    'return_quantity': 'sum',
    'item_refund_total': 'sum'
}).reset_index()

# Cruzando os dados
df_analise = pd.merge(vendas_sku, dev_sku, on='sku', how='inner')
df_analise['taxa_devolucao_pct'] = (df_analise['return_quantity'] / df_analise['qty_vendida']) * 100

# Filtrando apenas produtos com amostragem relevante (> 50 unidades vendidas no histórico)
df_analise = df_analise[df_analise['qty_vendida'] > 50]

viloes = df_analise.sort_values('taxa_devolucao_pct', ascending=False).head(10)

display(viloes.style.format({
    'taxa_devolucao_pct': '{:.2f}%',
    'item_refund_total': 'R$ {:,.2f}'
}))

### 3. Exemplo Prático do Framework de Decisão

**Fato Observado:** 
*(Substituir pelas conclusões visuais dos gráficos acima)*

**Hipótese:**
*(Substituir)*

**Recomendação:**
*(Substituir)*